# Week 29, Day 1: Ions, gradients and the Nernst potential

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w029/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The four Nernst potentials

Write `nernst(c_out, c_in, z, T=310.15)` returning the Nernst potential in millivolts, then fill the dictionary `E` with the potentials of potassium, sodium, chloride and calcium at the concentrations in the table.

In [ ]:
import numpy as np
R, F = 8.314, 96485.0            # J/(mol K), C/mol

def nernst(c_out, c_in, z, T=310.15):
    return ...

conc = {"K": (5, 140, 1), "Na": (145, 12, 1), "Cl": (110, 7, -1), "Ca": (2, 1e-4, 2)}
E = {ion: ... for ion, (c_out, c_in, z) in conc.items()}
for ion, v in E.items():
    print(f"E_{ion:2s} = {v:+7.1f} mV")
print(f"squid axon at 6.3 C: E_K = {nernst(20, 400, 1, 279.45):.1f} mV")

In [ ]:
import numpy as np
def test_function():
    "nernst gives (RT/zF) ln(out/in) in millivolts"
    assert abs(nernst(5, 140, 1) - 1000 * 8.314 * 310.15 / 96485 * np.log(5 / 140)) < 1e-9
    assert abs(nernst(110, 7, -1) + 1000 * 8.314 * 310.15 / 96485 * np.log(110 / 7)) < 1e-9

def test_values():
    "E holds the four potentials: potassium near -89 mV, sodium near +67, chloride near -74, calcium near +132"
    assert abs(E["K"] + 89.1) < 0.5 and abs(E["Na"] - 66.6) < 0.5 and abs(E["Cl"] + 73.6) < 0.5 and abs(E["Ca"] - 132.3) < 0.5

test_function()
test_values()
print("All checks passed")

### Exercise: Diffusion plus drift gives Boltzmann

Write the update inside the loop: drift by -dV * dt plus a Gaussian kick with standard deviation sqrt(2 dt), then reflect at 0 (take the absolute value) and at 1 (replace x by 2 - x where x exceeds 1). Then fit a straight line to the log of the histogram and store its slope in `slope`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(29)
dV, dt = 2.0, 2e-4                         # potential step in units of V_T; time step with D = 1
x = r.uniform(0, 1, 10000)
for step in range(3000):
    x = x                                  # replace with drift, diffusion and reflection
counts, edges = np.histogram(x, 20, (0, 1))
centres = (edges[1:] + edges[:-1]) / 2
slope = ...
print(f"fitted slope {slope:.2f}; Boltzmann predicts {-dV:.2f}")
print(f"density ratio left/right {counts[0] / counts[-1]:.1f}; exp(dV) = {np.exp(dV):.1f}")
plt.semilogy(centres, counts, "o"); plt.xlabel("position across the box"); plt.ylabel("ions per bin"); plt.show()

In [ ]:
import numpy as np
def test_reflect():
    "all particles stay inside the box"
    assert x.min() >= 0 and x.max() <= 1

def test_slope():
    "the log density falls linearly with slope close to -dV (within 0.2)"
    assert abs(slope + dV) < 0.2

test_reflect()
test_slope()
print("All checks passed")

### Exercise: Counting the charge

For spheres of diameter 20 μm, 1 μm and 0.2 μm (the last two like a thin dendrite or axon), compute the number of monovalent ions needed to charge the membrane to 70 mV, `n_moved`, and the number of potassium ions inside at 140 mM, `n_inside`. Work in centimetres: area πd², volume πd³/6 cm³, which is πd³/6000 litres.

In [ ]:
import numpy as np
e, NA, c_m = 1.602e-19, 6.022e23, 1e-6       # C, per mol, F per cm^2
d = np.array([20e-4, 1e-4, 0.2e-4])          # diameters in cm
n_moved = ...
n_inside = ...
for di, m, k in zip(d * 1e4, n_moved, n_inside):
    print(f"d = {di:5.1f} um: {m:9.3g} ions to charge, {k:9.3g} K+ inside, fraction {m / k:.1e}")

In [ ]:
import numpy as np
def test_counts():
    "n_moved is C V / e and n_inside is the potassium in the sphere"
    d = 20e-4
    assert abs(n_moved[0] - 1e-6 * np.pi * d**2 * 0.07 / 1.602e-19) / n_moved[0] < 1e-6
    assert abs(n_inside[0] - 0.14 * np.pi / 6 * d**3 / 1000 * 6.022e23) / n_inside[0] < 1e-6

def test_scaling():
    "the fraction moved grows as the cell shrinks, inversely with diameter"
    f = n_moved / n_inside
    assert f[0] < 1e-4 and abs(f[2] / f[0] - 100) < 1e-6

test_counts()
test_scaling()
print("All checks passed")